# Notebook 01 — EDA & Preprocessing

**Goal:** Explore `clinical_raw.csv`, encode categoricals, drop leaky columns,
produce Table 1 (baseline characteristics), and save `clinical_clean.csv`.

**Key decisions (protocol doc2):**
- Age: ordinal <20=0, 21-40=1, 41-60=2, >60=3
- BMI: ordinal <18.5=0, 18.5-25=1, 25-30=2, >30=3
- Binary yes/no → 1/0 (preserve NaN)
- `ttdeath_3M` and `dataset`/`cohort` dropped (leaky or admin)
- XGBoost handles NaN natively — no imputation here
- SimpleImputer for LR surrogate handled in notebook 05

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, str(Path("../src").resolve()))
from data_loader import load_clinical_raw, PROCESSED_DIR

FIG_DIR = Path("../../results/figures")
TAB_DIR = Path("../../results/tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

df = load_clinical_raw()
print(f"Loaded clinical_raw: {df.shape}")

Loaded clinical_raw: (413, 25)


## 1. Outcome variable

In [2]:
TARGET = "evdeath_3M"
y = df[TARGET].astype(int)
print(f"Died (1): {y.sum()}   Survived (0): {(y==0).sum()}   Total: {len(y)}")
print(f"3-month mortality rate: {y.mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
counts = y.value_counts().sort_index()
bars = ax.bar(["Survived (0)", "Died (1)"], counts.values,
              color=["steelblue", "firebrick"], edgecolor="k", width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("3-month mortality distribution", fontsize=12)
ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_outcome_distribution.png", dpi=150)
plt.show()
print("Saved: 01_outcome_distribution.png")

Died (1): 99   Survived (0): 314   Total: 413
3-month mortality rate: 24.0%
Saved: 01_outcome_distribution.png


/tmp/ipykernel_176719/3752082523.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Encode categorical features

In [3]:
AGE_ORDER = {"<20": 0, "21-40": 1, "41-60": 2, ">60": 3}
BMI_ORDER = {"<18.5": 0, "18.5-25": 1, "25-30": 2, ">30": 3}
DIAG_ORDER = {"possible TBM": 0, "probable TBM": 1, "definite TBM": 2}

def encode_binary(col: pd.Series) -> pd.Series:
    """yes→1, no→0; preserve NaN."""
    return col.str.lower().map({"yes": 1, "no": 0})

df_enc = df.copy()

if "AGE" in df_enc.columns:
    df_enc["AGE"] = df_enc["AGE"].map(AGE_ORDER)
    print("AGE value counts:", df_enc["AGE"].value_counts().sort_index().to_dict())

if "BMI" in df_enc.columns:
    df_enc["BMI"] = df_enc["BMI"].map(BMI_ORDER)
    print("BMI value counts:", df_enc["BMI"].value_counts().sort_index().to_dict())

if "DIAGNOSTIC_SCORE" in df_enc.columns:
    df_enc["DIAGNOSTIC_SCORE"] = df_enc["DIAGNOSTIC_SCORE"].map(DIAG_ORDER)
    print("DIAGNOSTIC_SCORE:", df_enc["DIAGNOSTIC_SCORE"].value_counts().to_dict())

binary_cols = ["SEX", "HIV", "History_TBT", "cavity",
               "MYCORESULT", "GeneXpert", "ZNSMEAR", "ART_BL"]
for col in binary_cols:
    if col in df_enc.columns:
        df_enc[col] = encode_binary(df_enc[col].astype(str))

print("\nEncoding complete. dtypes:")
print(df_enc.dtypes.to_string())

AGE value counts: {0.0: 10, 1.0: 176, 2.0: 154, 3.0: 66}
BMI value counts: {0.0: 127, 1.0: 260, 2.0: 22}
DIAGNOSTIC_SCORE: {2.0: 174, 1.0: 143, 0.0: 70}

Encoding complete. dtypes:
SEX                 float64
AGE                 float64
HIV                   int64
DIAGNOSTIC_SCORE    float64
TBMGRADE              int64
WBC_x               float64
totalneutrobl_x     float64
totallymphobl_x     float64
CSFWBC              float64
totalcsfneutro      float64
totalcsflympho      float64
evdeath_3M            int64
ttdeath_3M            int64
dataset              object
WBC_y               float64
totalneutrobl_y     float64
totallymphobl_y     float64
BMI                 float64
symp_dur            float64
History_TBT         float64
GCS                 float64
MYCORESULT          float64
GeneXpert           float64
ZNSMEAR             float64
cohort                int64


## 3. Drop leaky & admin columns

In [4]:
LEAKY = ["ttdeath_3M", "dataset", "cohort", "HIV_viralload"]
drop = [c for c in LEAKY if c in df_enc.columns]
print(f"Dropping leaky/admin columns: {drop}")
df_enc = df_enc.drop(columns=drop)
print(f"Shape: {df_enc.shape}")

Dropping leaky/admin columns: ['ttdeath_3M', 'dataset', 'cohort']
Shape: (413, 22)


## 4. Remaining missingness summary

In [5]:
miss = df_enc.isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print("Features with remaining missingness:")
print(miss_nonzero.to_string())

Features with remaining missingness:
totalcsfneutro      0.571429
MYCORESULT          0.300242
GeneXpert           0.220339
ZNSMEAR             0.087167
CSFWBC              0.065375
totalcsflympho      0.065375
DIAGNOSTIC_SCORE    0.062954
AGE                 0.016949
totallymphobl_y     0.014528
WBC_x               0.014528
WBC_y               0.014528
totalneutrobl_y     0.014528
totalneutrobl_x     0.014528
totallymphobl_x     0.014528
History_TBT         0.012107
BMI                 0.009685
SEX                 0.004843
GCS                 0.004843
symp_dur            0.002421


## 5. Table 1 — Baseline characteristics

In [6]:
survived = df_enc[df_enc[TARGET] == 0]
died = df_enc[df_enc[TARGET] == 1]

def _med_iqr(s: pd.Series) -> str:
    s = s.dropna()
    if len(s) == 0:
        return "NA"
    return f"{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]"

def _n_pct(s: pd.Series) -> str:
    n = s.notna().sum()
    pos = s.sum()
    return f"{pos:.0f} ({pos/n*100:.1f}%)" if n > 0 else "NA"

cont_cols = [c for c in ["AGE", "GCS", "WBC", "totalneutrobl", "totallymphobl",
                          "CSFWBC", "totalcsfneutro", "totalcsflympho",
                          "symp_dur", "BMI", "CD4"] if c in df_enc.columns]
bin_cols  = [c for c in ["SEX", "HIV", "DIAGNOSTIC_SCORE", "TBMGRADE",
                          "History_TBT", "MYCORESULT", "GeneXpert", "ZNSMEAR"]
             if c in df_enc.columns]

rows = []
for col in cont_cols:
    rows.append({"Variable": col, "All": _med_iqr(df_enc[col]),
                 "Survived": _med_iqr(survived[col]),
                 "Died": _med_iqr(died[col]), "Type": "median [IQR]"})
for col in bin_cols:
    rows.append({"Variable": col, "All": _n_pct(df_enc[col]),
                 "Survived": _n_pct(survived[col]),
                 "Died": _n_pct(died[col]), "Type": "n (%)"})

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))
table1.to_csv(TAB_DIR / "01_table1_baseline.csv", index=False)
print("\nSaved: results/tables/01_table1_baseline.csv")

        Variable                All           Survived              Died         Type
             AGE      2.0 [1.0–2.0]      2.0 [1.0–2.0]     2.0 [1.0–3.0] median [IQR]
             GCS   14.0 [13.0–15.0]   15.0 [13.0–15.0]  13.0 [10.0–14.0] median [IQR]
          CSFWBC 131.5 [23.2–343.8] 140.0 [31.0–352.0] 82.0 [15.0–304.0] median [IQR]
  totalcsfneutro  51.9 [17.1–164.0]  53.9 [20.2–153.2]  50.6 [4.2–173.1] median [IQR]
  totalcsflympho 101.2 [19.2–226.9] 109.5 [28.1–248.0]  60.8 [8.0–157.8] median [IQR]
        symp_dur   15.5 [11.0–22.0]   15.0 [11.0–20.0]  18.0 [11.5–30.0] median [IQR]
             BMI      1.0 [0.0–1.0]      1.0 [0.0–1.0]     1.0 [0.0–1.0] median [IQR]
             SEX        267 (65.0%)        208 (66.7%)        59 (59.6%)        n (%)
             HIV         74 (17.9%)         47 (15.0%)        27 (27.3%)        n (%)
DIAGNOSTIC_SCORE       491 (126.9%)       366 (126.6%)      125 (127.6%)        n (%)
        TBMGRADE       706 (170.9%)       490 (156.1%)

## 6. Distribution plots for key predictors

In [7]:
key_cont = [c for c in ["GCS", "WBC", "CSFWBC", "totalneutrobl"] if c in df_enc.columns]
if key_cont:
    fig, axes = plt.subplots(1, len(key_cont), figsize=(4*len(key_cont), 4))
    if len(key_cont) == 1:
        axes = [axes]
    for ax, col in zip(axes, key_cont):
        for label, grp, color in [(0, survived, "steelblue"), (1, died, "firebrick")]:
            sns.kdeplot(grp[col].dropna(), ax=ax, label=f"evdeath={label}",
                        color=color, fill=True, alpha=0.3)
        ax.set_title(col)
        ax.legend(fontsize=8)
    fig.suptitle("Key predictors by outcome", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "01_predictor_distributions.png", dpi=150)
    plt.show()
    print("Saved: 01_predictor_distributions.png")

Saved: 01_predictor_distributions.png


/tmp/ipykernel_176719/966280948.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Correlation heatmap

In [8]:
num_cols = df_enc.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore").tolist()
fig, ax = plt.subplots(figsize=(max(8, len(num_cols)), max(7, len(num_cols)-1)))
corr = df_enc[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.4, ax=ax)
ax.set_title("Feature correlation heatmap", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_correlation_heatmap.png", dpi=150)
plt.show()
print("Saved: 01_correlation_heatmap.png")

Saved: 01_correlation_heatmap.png


/tmp/ipykernel_176719/4056604010.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save clinical_clean.csv

In [9]:
out_path = PROCESSED_DIR / "clinical_clean.csv"
df_enc.to_csv(out_path, index=False)
print(f"Saved: {out_path}  shape={df_enc.shape}")
print(f"Columns: {df_enc.columns.tolist()}")
df_enc.head()

Saved: /home/jeremy-mboe/Documents/Kuliah/Sem4/PnD/TB_Meningitis/dataset/processed/clinical_clean.csv  shape=(413, 22)
Columns: ['SEX', 'AGE', 'HIV', 'DIAGNOSTIC_SCORE', 'TBMGRADE', 'WBC_x', 'totalneutrobl_x', 'totallymphobl_x', 'CSFWBC', 'totalcsfneutro', 'totalcsflympho', 'evdeath_3M', 'WBC_y', 'totalneutrobl_y', 'totallymphobl_y', 'BMI', 'symp_dur', 'History_TBT', 'GCS', 'MYCORESULT', 'GeneXpert', 'ZNSMEAR']


,SEX,AGE,HIV,DIAGNOSTIC_SCORE,TBMGRADE,WBC_x,totalneutrobl_x,totallymphobl_x,CSFWBC,totalcsfneutro,...,WBC_y,totalneutrobl_y,totallymphobl_y,BMI,symp_dur,History_TBT,GCS,MYCORESULT,GeneXpert,ZNSMEAR
0,1.0,1.0,0,2.0,2,8.30,6.83920,0.56440,40.0,8.00,...,8.30,6.83920,0.56440,1.0,33.0,0.0,11.0,NaN,1.0,0.0
1,1.0,3.0,0,1.0,2,10.20,5.92620,1.20360,52.0,NaN,...,10.20,5.92620,1.20360,0.0,17.0,0.0,12.0,0.0,0.0,0.0
2,1.0,1.0,1,1.0,2,23.69,21.06041,0.52118,133.0,51.87,...,23.69,21.06041,0.52118,0.0,40.0,0.0,11.0,NaN,NaN,NaN
3,1.0,1.0,0,0.0,2,7.70,5.22060,1.77100,1908.0,NaN,...,7.70,5.22060,1.77100,1.0,16.0,0.0,14.0,NaN,NaN,0.0
4,0.0,2.0,0,1.0,1,9.42,7.34760,1.06446,492.0,236.16,...,9.42,7.34760,1.06446,0.0,15.0,0.0,14.0,0.0,0.0,0.0
